# NF-UNSW-NB15-v3 — Phase 3: data preprocessing (cleaning)

**Owner:** Ahmed Hamouda. Applies the approved decisions of phase 2 for this dataset
(`report/data_problems/phase2_data_problems.pdf`, section 1 and section 6). Phase 2 evidence: `01_unsw.ipynb`.

| Step | What | Decision |
|---|---|---|
| 0 | Load the raw CSV | – |
| 1 | Drop the two `..._SECOND_BYTES` columns | U2 |
| 2 | Split `L7_PROTO` into `L7_MAIN` + `L7_SUB` | U3 |
| 3 | Remove exact duplicate rows | U4 |
| 4 | One row per (behaviour, label), with a `count` column | Section 6 |
| 5 | Shared `clean()`: drop the IP columns, NaN / inf, duplicates | M3 + shared rule |
| 6 | `map_labels()`: `attack_label`, `attack_family`, `is_attack`, `source_dataset`; then drop `Label` | U5, U6 |
| 7 | Checks after cleaning | – |
| 8 | Save the cleaned file and the before/after tables | – |

**Not done here** (done once on the merged data, identical for all four datasets): ICMP fields → 0 (M6), drop
`MIN/MAX_IP_PKT_LEN` (M7), drop `TCP_FLAGS` / `ICMP_IPV4_TYPE` (M8), DNS TTL above 2^31−1 → 0 (M9), codes → categorical
(M1). **Kept on purpose:** invalid codes (U7), small classes (U5), attack-vs-attack conflicts (U8), byte-count and
size-bin inconsistencies (K3, K4), outliers (K10), TTL = 0 (K1).

## Step 0 — Load the raw data

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv

from src.config import *

pd.set_option("display.max_columns", 100)
SOURCE = "UNSW"
path = RAW_DIR / DATASETS[SOURCE]["file"]

header = pd.read_csv(path, nrows=0).columns
types = {c: pa.int32() for c in header if c not in ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack"]}
for c in ["FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "DNS_TTL_ANSWER"]:
    types[c] = pa.int64()
for c in ["SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES"]:
    types[c] = pa.float64()
types["L7_PROTO"] = pa.string()   # read as text: the part after the dot is a second code (U3)
df = pv.read_csv(path, convert_options=pv.ConvertOptions(column_types=types)).to_pandas()

raw_labels = df["Attack"].value_counts()
steps = []
def log(step, decision, what):
    steps.append({"step": step, "decision": decision, "rows": len(df), "columns": df.shape[1], "what": what})
    print(f"{step}: {len(df):,} rows x {df.shape[1]} columns  ({what})")

log("0 load", "-", "raw file")
raw_labels

0 load: 2,365,424 rows x 55 columns  (raw file)


Attack
Benign            2237731
Exploits            42748
Fuzzers             33816
Generic             19651
Reconnaissance      17074
DoS                  5980
Backdoor             4659
Shellcode            2381
Analysis             1226
Worms                 158
Name: count, dtype: int64

## Step 1 — Drop the two `..._SECOND_BYTES` columns (U2)
They are mislabelled (bytes per millisecond, directions swapped), contain NaN and infinity whenever the duration is 0,
and their information is in `SRC_TO_DST_AVG_THROUGHPUT` / `DST_TO_SRC_AVG_THROUGHPUT`. They must go **before**
`clean()`, which deletes every row with a NaN (75.8% of Generic).

In [2]:
sb = ["SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES"]
num = df.select_dtypes("number")
print("before: NaN cells", int(df.isna().sum().sum()), "| inf cells", int(np.isinf(num).sum().sum()),
      "| rows that clean() would drop:", int((df[sb].isna() | np.isinf(df[sb])).any(axis=1).sum()))
df = df.drop(columns=sb)
num = df.select_dtypes("number")
print("after:  NaN cells", int(df.isna().sum().sum()), "| inf cells", int(np.isinf(num).sum().sum()))
log("1 drop SECOND_BYTES", "U2", "2 columns removed")

before: NaN cells 63425 | inf cells 181561 | rows that clean() would drop: 122493


after:  NaN cells 0 | inf cells 0
1 drop SECOND_BYTES: 2,365,424 rows x 53 columns  (2 columns removed)


## Step 2 — Split `L7_PROTO` into `L7_MAIN` and `L7_SUB` (U3)
`7.37` = HTTP (7) carrying BitTorrent (37). The column is read as text so the second code is not lost; `L7_SUB` = 0
when there is no dot. CIC-2018 and ToN-IoT get the same two columns (with `L7_SUB` = 0), so the merged columns match.

In [3]:
parts = df["L7_PROTO"].str.split(".", n=1, expand=True)
pos = df.columns.get_loc("L7_PROTO")
df.insert(pos, "L7_MAIN", parts[0].astype("int32"))
df.insert(pos + 1, "L7_SUB", parts[1].fillna("0").astype("int32"))
has_sub = parts[1].notna() & (parts[1] != "0")
one_digit = has_sub & (parts[1].str.len() == 1)
print(f"flows with a sub-code: {has_sub.sum():,} | sub-codes with one digit (a lost trailing zero is possible, e.g. 7.5): {one_digit.sum():,}")
display(df.loc[has_sub, ["L7_PROTO", "L7_MAIN", "L7_SUB"]].value_counts().head(6))
df = df.drop(columns="L7_PROTO")
log("2 split L7_PROTO", "U3", "L7_PROTO -> L7_MAIN + L7_SUB")

flows with a sub-code: 23,866 | sub-codes with one digit (a lost trailing zero is possible, e.g. 7.5): 8


L7_PROTO  L7_MAIN  L7_SUB
7.37      7        37        18765
10.16     10       16         4877
7.253     7        253         146
7.376     7        376          45
7.5       7        5             8
7.212     7        212           7
Name: count, dtype: int64

2 split L7_PROTO: 2,365,424 rows x 54 columns  (L7_PROTO -> L7_MAIN + L7_SUB)


## Step 3 — Remove exact duplicate rows (U4)
Rows identical in **every** column (including IP addresses and the millisecond timestamp) are the same flow written
twice. They are removed before step 4, so that `count` counts real flows, not copies.

In [4]:
dup = df.duplicated()
print(f"exact duplicates: {dup.sum():,}")
display(df.loc[dup, "Attack"].value_counts().rename("removed"))
df = df[~dup]
log("3 exact duplicates", "U4", f"{int(dup.sum()):,} copies removed")

exact duplicates: 14,815


Attack
Benign      14801
DoS             9
Exploits        4
Backdoor        1
Name: removed, dtype: int64

3 exact duplicates: 2,350,609 rows x 54 columns  (14,815 copies removed)


## Step 4 — One row per (behaviour, label), with `count` (section 6)
A behaviour is everything the flow did, leaving out who and when (IP addresses, source port, timestamps, DNS query ID)
and the columns removed after the merge. Repeats of the same behaviour with the same label are collapsed into their
first row; `count` keeps how many flows it stands for. Every different behaviour is kept. This runs **before** the IP
columns are dropped, so two machines doing the same thing are still counted as two flows.

In [5]:
flows_before = len(df)
df = collapse_behaviours(df, "Attack")
print(f"{flows_before:,} flows -> {len(df):,} rows; sum of count = {df['count'].sum():,} (must equal the flows)")
assert df["count"].sum() == flows_before
display(pd.DataFrame({"flows": raw_labels, "rows kept": df["Attack"].value_counts(),
                      "largest count": df.groupby("Attack")["count"].max()}).sort_values("flows", ascending=False))
log("4 collapse behaviours", "section 6", "repeats -> one row + count")

2,350,609 flows -> 1,405,100 rows; sum of count = 2,350,609 (must equal the flows)


,flows,rows kept,largest count
Attack,,,
Benign,2237731,1307351,201975
Exploits,42748,40424,929
Fuzzers,33816,26987,2125
Generic,19651,5726,12860
Reconnaissance,17074,11908,4384
DoS,5980,5360,316
Backdoor,4659,3656,256
Shellcode,2381,2316,58
Analysis,1226,1226,1


4 collapse behaviours: 1,405,100 rows x 55 columns  (repeats -> one row + count)


## Step 5 — Shared `clean()` (`src/config.py`)
Drops the identifier columns (the two IP addresses, M3), turns any inf into NaN, drops rows that still have a NaN, drops
duplicate rows, and makes the numbers smaller (`downcast`). After steps 1–4, no row should be lost here.

In [6]:
rows_before = len(df)
df, stats = clean(df, SOURCE)
print("rows lost in clean():", rows_before - len(df))
log("5 clean()", "M3 + shared rule", f"IP columns dropped; {rows_before - len(df)} rows lost")
stats

rows lost in clean(): 0
5 clean(): 1,405,100 rows x 53 columns  (IP columns dropped; 0 rows lost)


{'source_dataset': 'UNSW',
 'rows_raw': 1405100,
 'cols_raw': 55,
 'id_columns_dropped': 'IPV4_SRC_ADDR, IPV4_DST_ADDR',
 'inf_values': 0,
 'nan_values': 0,
 'imputed_columns': 'none',
 'rows_with_nan': 0,
 'duplicate_rows': 0,
 'rows_clean': 1405100,
 'cols_clean': 53}

## Step 6 — Shared labels (`map_labels`) and drop `Label`
Adds `attack_label` (clean name), `attack_family` (one of the 9 families), `is_attack` (0/1) and `source_dataset`.
`Label` is identical to `is_attack`, so it is dropped after the check; the raw `Attack` name is kept for traceability.
None of these columns is ever a model feature.

In [7]:
df = map_labels(df, "Attack", SOURCE)
assert (df["is_attack"].astype(int) == df["Label"].astype(int)).all(), "Label and is_attack disagree"
df = df.drop(columns="Label")
display(df.groupby(["attack_family", "attack_label"], observed=True)["count"].agg(rows="size", flows="sum"))
log("6 map_labels + drop Label", "U5, U6", "4 label columns added, Label dropped")

rows    flows
attack_family              attack_label                    
Benign                     Benign          1307351  2222930
Denial of service          DoS                5360     5971
Exploitation and intrusion Exploits          40424    42744
                           Shellcode          2316     2381
Fuzzing and crypto         Fuzzers           26987    33816
                           Generic            5726    19651
Malware and botnets        Backdoor           3656     4658
                           Worms               146      158
Reconnaissance             Analysis           1226     1226
                           Reconnaissance    11908    17074

6 map_labels + drop Label: 1,405,100 rows x 56 columns  (4 label columns added, Label dropped)


## Step 7 — Checks after cleaning

In [8]:
checks = validate(df)
print("validate(): total violations", int(checks["violations"].sum()))
num = df.select_dtypes("number")
print("NaN cells:", int(df.isna().sum().sum()), "| inf cells:", int(np.isinf(num).sum().sum()))
print("IP columns left:", [c for c in df.columns if "ADDR" in c])
print("flows represented:", f"{int(df['count'].sum()):,}", "| raw flows:", f"{int(raw_labels.sum()):,}",
      "| exact duplicates removed:", f"{int(raw_labels.sum() - df['count'].sum()):,}")
print(f"final: {len(df):,} rows x {df.shape[1]} columns")
df.dtypes.value_counts()

validate(): total violations 0


NaN cells: 0 | inf cells: 0
IP columns left: []
flows represented: 2,350,609 | raw flows: 2,365,424 | exact duplicates removed: 14,815
final: 1,405,100 rows x 56 columns


int32     48
int64      3
str        2
object     2
int8       1
Name: count, dtype: int64

## Step 8 — Save
`data/processed/UNSW_clean.parquet` (the cleaned table), `tables/UNSW_quality.csv` (one row per step) and
`tables/UNSW_labels.csv` (per label: raw flows, rows kept, flows represented).

In [9]:
log("8 final", "-", "cleaned table")
lab_map = map_labels(pd.DataFrame({"Attack": raw_labels.index}), "Attack", SOURCE).set_index("Attack")["attack_label"]
labels = cleaning_report(steps, raw_labels.groupby(lab_map).sum(), df, SOURCE)
out = save_clean(df, SOURCE)
print("saved", out.relative_to(ROOT), f"({out.stat().st_size / 1e6:.0f} MB)")
display(pd.DataFrame(steps))
labels

8 final: 1,405,100 rows x 56 columns  (cleaned table)


saved data\processed\UNSW_clean.parquet (76 MB)


,step,decision,rows,columns,what
0,0 load,-,2365424,55,raw file
1,1 drop SECOND_BYTES,U2,2365424,53,2 columns removed
2,2 split L7_PROTO,U3,2365424,54,L7_PROTO -> L7_MAIN + L7_SUB
3,3 exact duplicates,U4,2350609,54,"14,815 copies removed"
4,4 collapse behaviours,section 6,1405100,55,repeats -> one row + count
5,5 clean(),M3 + shared rule,1405100,53,IP columns dropped; 0 rows lost
6,6 map_labels + drop Label,"U5, U6",1405100,56,"4 label columns added, Label dropped"
7,8 final,-,1405100,56,cleaned table


,source_dataset,flows_raw,rows_clean,flows_represented,attack_family
attack_label,,,,,
Analysis,UNSW,1226,1226,1226,Reconnaissance
Backdoor,UNSW,4659,3656,4658,Malware and botnets
Benign,UNSW,2237731,1307351,2222930,Benign
DoS,UNSW,5980,5360,5971,Denial of service
Exploits,UNSW,42748,40424,42744,Exploitation and intrusion
Fuzzers,UNSW,33816,26987,33816,Fuzzing and crypto
Generic,UNSW,19651,5726,19651,Fuzzing and crypto
Reconnaissance,UNSW,17074,11908,17074,Reconnaissance
Shellcode,UNSW,2381,2316,2381,Exploitation and intrusion
